# Group A1: Lab Test – Padding Masking, Transformer Encoder, and Test Set Evaluation

### Task Objectives:
1. **Preprocessing**: Implement tokenization and boolean padding mask generation (`True` for `<PAD>`).
2. **Model Architecture**: Construct `TransformerEncoderClassifier` using `nn.TransformerEncoder` + Mean Pooling.
3. **Evaluation Loop**: Implement evaluation logic to check test set loss and accuracy.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

raw_train_reviews = [
    ("fantastic wonderful great cinema loved it", 1),
    ("superb brilliant masterpiece highly recommend", 1),
    ("terrible awful boring script disaster", 0),
    ("dull bad horrible direction waste time", 0)
]

raw_test_reviews = [
    ("wonderful brilliant movie loved it", 1),
    ("terrible boring direction waste time", 0)
]

# Pre-coded Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# =============================================================================
# PART 1: PREPROCESSING & MASKING (Student TODO)
# =============================================================================
def build_vocab(corpus):
    """
    TODO: Build a dictionary mapping unique words to integer IDs.
    Must reserve: '<PAD>': 0, '<UNK>': 1
    """
    vocab = {"<PAD>": 0, "<UNK>": 1}
    # YOUR CODE HERE
    pass

def process_batch(sentences, vocab, max_len=8):
    """
    TODO:
    1. Tokenize and pad list of sentences to max_len.
    2. Create boolean padding mask tensor where True corresponds to padding index 0.
    Returns: input_ids (tensor long), padding_mask (tensor bool)
    """
    # YOUR CODE HERE
    pass

# =============================================================================
# PART 2: MODEL ARCHITECTURE (Student TODO)
# =============================================================================
class TransformerEncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=2, num_layers=1, num_classes=1, max_len=8):
        super().__init__()
        self.d_model = d_model
        # =========================================================================
        # TODO: Define layers:
        # 1. Embedding (padding_idx=0)
        # 2. PositionalEncoding
        # 3. TransformerEncoderLayer and TransformerEncoder (batch_first=True)
        # 4. Linear layer (d_model -> num_classes)
        # =========================================================================
        # YOUR CODE HERE
        pass

    def forward(self, x, padding_mask=None):
        # =========================================================================
        # TODO: Implement forward pass with Mean Pooling across dim=1
        # Pass src_key_padding_mask into transformer_encoder
        # =========================================================================
        # YOUR CODE HERE
        pass

# =============================================================================
# PART 3: EVALUATION LOOP (Student TODO)
# =============================================================================
def evaluate_transformer(model, loader, criterion):
    """
    TODO: Evaluate model over loader. Return average loss and accuracy percentage.
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        # YOUR CODE HERE
        pass
    return (total_loss / len(loader)), (correct / total) * 100

# =============================================================================
# PART 4: AUTOMATED VERIFICATION SUITE (Do Not Modify)
# =============================================================================
def run_test():
    vocab = build_vocab([t for t, _ in raw_train_reviews])
    
    train_x, train_mask = process_batch([t for t, _ in raw_train_reviews], vocab)
    train_y = torch.tensor([l for _, l in raw_train_reviews], dtype=torch.float32)
    
    test_x, test_mask = process_batch([t for t, _ in raw_test_reviews], vocab)
    test_y = torch.tensor([l for _, l in raw_test_reviews], dtype=torch.float32)
    
    train_loader = DataLoader(TensorDataset(train_x, train_mask, train_y), batch_size=2, shuffle=True)
    test_loader = DataLoader(TensorDataset(test_x, test_mask, test_y), batch_size=2)
    
    model = TransformerEncoderClassifier(vocab_size=len(vocab)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.BCEWithLogitsLoss()
    
    model.train()
    for epoch in range(25):
        for bx, bm, by in train_loader:
            bx, bm, by = bx.to(device), bm.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx, padding_mask=bm), by)
            loss.backward()
            optimizer.step()
            
    test_loss, test_acc = evaluate_transformer(model, test_loader, criterion)
    print(f"Transformer Test Set Loss: {test_loss:.4f} | Accuracy: {test_acc:.2f}%")
    assert test_acc == 100.0, "Verification Failed: Test accuracy did not reach target."
    print("Lab task passed!")

run_test()